### CNN v1

* first 300s
* hjort1/2+power band
* 5x5
* 8s windows
* all channels

In [1]:
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from scipy import signal
from scipy.integrate import simpson

SEED = 1738
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def r2(y, p): return 1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2)

def metrics(log_pred, age):
    pred = np.exp(log_pred)
    return {'RMSE': np.sqrt(np.mean((pred - age) ** 2)), 'MAE': np.mean(np.abs(pred - age)), 'R2 (age)': r2(age, pred), 'R2 (log age)': r2(np.log(age), log_pred)}

In [2]:
loss_fn = nn.MSELoss()

In [3]:
DATA = Path('eeg_starter/data')
FS, WIN_S, REGION_S, C = 64, 8, 300, 20
WIN, N_WIN = WIN_S * FS, REGION_S // WIN_S
BANDS = [(0.5, 4), (4, 8), (8, 12), (12, 30)]

def handcrafted(eeg):
    w = eeg[:, :REGION_S * FS].double().numpy().reshape(C, REGION_S // 30, 30 * FS)
    var_x, var_dx = w.var(-1), np.diff(w, axis=-1).var(-1)
    f, psd = signal.welch(w, FS, nperseg=4 * FS, axis=-1)
    bp = [simpson(psd[..., (f >= lo) & (f <= hi)], dx=f[1] - f[0], axis=-1) for lo, hi in BANDS]
    with np.errstate(all='ignore'):
        feats = np.stack([np.log(var_x), np.sqrt(var_dx / var_x), *np.log(bp)])
    feats[:, var_x < 1e-6] = np.nan
    return np.nanmedian(feats, axis=-1).ravel()

def load_split(name):
    seg = torch.load(DATA / f'segments_h1k_{name}.pt')
    eeg = [torch.load(DATA / 'pre_processed' / s[4].replace('.edf', '.pt')) for s in seg]
    windows = torch.stack([e[:, :N_WIN * WIN].unflatten(1, (N_WIN, WIN)).transpose(0, 1) for e in eeg])
    return windows, np.stack([handcrafted(e) for e in eeg]), np.array([s[3] for s in seg])

X_tr, H_tr, age_tr = load_split('train')
X_va, H_va, age_va = load_split('val')
mu, sd = np.nanmean(H_tr, 0), np.nanstd(H_tr, 0)
H_tr, H_va = [torch.tensor(np.nan_to_num((H - mu) / sd), dtype=torch.float32) for H in (H_tr, H_va)]
print(X_tr.shape, H_tr.shape, X_va.shape, H_va.shape)

torch.Size([825, 37, 20, 512]) torch.Size([825, 120]) torch.Size([103, 37, 20, 512]) torch.Size([103, 120])


In [4]:
F1, F2, K, POOL = 16, 32, 5, 2

class CNNv1(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(C, C * F1, K, groups=C), nn.ReLU(), # group=C - paralell
            nn.Conv1d(C * F1, C * F2, K, groups=C), nn.ReLU(),
            nn.MaxPool1d(POOL),
            nn.AdaptiveAvgPool1d(1), nn.Flatten())
        self.fc = nn.Linear(C * F2 + H_tr.shape[1], 1)

    def forward(self, x, hand):
        return self.fc(torch.cat([self.conv(x), hand], 1)).squeeze(1)

model = CNNv1().to(DEVICE)
print(sum(p.numel() for p in model.parameters()), 'parameters')

54521 parameters


In [6]:
EPOCHS, BATCH = 20, 64
opt = torch.optim.Adam(model.parameters())

def predict(X, H):
    model.eval()
    with torch.no_grad():
        return np.array([model(x.float().to(DEVICE), h.expand(N_WIN, -1).to(DEVICE)).mean().item() for x, h in zip(X, H)])

y_tr = torch.tensor(np.log(age_tr), dtype=torch.float32)
child = torch.arange(len(X_tr)).repeat_interleave(N_WIN)
win = torch.arange(N_WIN).repeat(len(X_tr))
for epoch in range(EPOCHS):
    model.train()
    for b in torch.randperm(len(child)).split(BATCH):
        c, w = child[b], win[b]
        loss = loss_fn(model(X_tr[c, w].float().to(DEVICE), H_tr[c].to(DEVICE)), y_tr[c].to(DEVICE))
        opt.zero_grad(); loss.backward(); opt.step()
    print(epoch + 1, {k: round(float(v), 3) for k, v in metrics(predict(X_va, H_va), age_va).items()})

1 {'RMSE': 1.682, 'MAE': 1.109, 'R2 (age)': 0.821, 'R2 (log age)': 0.868}
2 {'RMSE': 1.736, 'MAE': 1.132, 'R2 (age)': 0.809, 'R2 (log age)': 0.842}
3 {'RMSE': 1.841, 'MAE': 1.19, 'R2 (age)': 0.785, 'R2 (log age)': 0.836}
4 {'RMSE': 1.592, 'MAE': 1.005, 'R2 (age)': 0.84, 'R2 (log age)': 0.88}
5 {'RMSE': 1.588, 'MAE': 1.001, 'R2 (age)': 0.84, 'R2 (log age)': 0.882}
6 {'RMSE': 1.61, 'MAE': 1.031, 'R2 (age)': 0.836, 'R2 (log age)': 0.898}
7 {'RMSE': 1.473, 'MAE': 0.907, 'R2 (age)': 0.863, 'R2 (log age)': 0.886}
8 {'RMSE': 1.602, 'MAE': 1.003, 'R2 (age)': 0.838, 'R2 (log age)': 0.896}
9 {'RMSE': 1.562, 'MAE': 0.933, 'R2 (age)': 0.846, 'R2 (log age)': 0.881}
10 {'RMSE': 1.51, 'MAE': 0.9, 'R2 (age)': 0.856, 'R2 (log age)': 0.891}
11 {'RMSE': 1.467, 'MAE': 0.9, 'R2 (age)': 0.864, 'R2 (log age)': 0.89}
12 {'RMSE': 1.536, 'MAE': 0.941, 'R2 (age)': 0.851, 'R2 (log age)': 0.891}
13 {'RMSE': 1.572, 'MAE': 0.97, 'R2 (age)': 0.844, 'R2 (log age)': 0.901}
14 {'RMSE': 1.551, 'MAE': 0.945, 'R2 (age)': 0